# Complete Guide to Sandbox Executor

The `SandboxExecutor` is a critical component in AlgoDisco for safely executing untrusted code. It runs code in isolated subprocesses with timeout protection and resource limits, preventing malicious or buggy code from affecting the host system.

## Table of Contents
1. [Core Concepts](#1-core-concepts)
2. [Initialization Parameters](#2-initialization-parameters)
3. [Execution Methods](#3-execution-methods)
4. [Usage Examples](#4-usage-examples)
5. [Security Best Practices](#5-security-best-practices)
6. [Integration with Evaluator](#6-integration-with-evaluator)
7. [Summary](#7-summary)


## 1. Core Concepts

Understanding the fundamental concepts behind sandboxing is essential for safe code execution in algorithm discovery workflows.

In [ ]:
# Sandbox Core Features:
# 1. Process Isolation - Code runs in an independent subprocess
#    This ensures that any crashes, infinite loops, or memory leaks
#    in the executed code do not affect the parent process.
# 2. Timeout Protection - Prevents runaway code from executing forever
#    Critical for algorithm discovery where generated code may contain bugs
# 3. Resource Limits - Controls memory usage to prevent OOM scenarios
#    Shared memory is limited to prevent resource exhaustion attacks
# 4. Output Redirection - Capture or discard stdout/stderr
#    Useful when you want to suppress verbose output from executed code

from algodisco.toolkit.sandbox import SandboxExecutor
from algodisco.base.evaluator import EvalResult

## 2. Initialization Parameters

The `SandboxExecutor` constructor accepts several parameters to configure the sandbox environment. Understanding these parameters is crucial for balancing security, performance, and functionality.

In [ ]:
# SandboxExecutor Initialization Parameters:

# evaluate_worker: REQUIRED - The work object to execute
#                  Must have callable methods that will be invoked
# recur_kill_eval_proc: Optional - If True, recursively kills all child
#                       processes when termination is needed. Default: False
# debug_mode: Optional - If True, enables verbose logging for debugging.
#             Default: False
# join_timeout_seconds: Optional - How long to wait for process completion.
#                        Default: 10 seconds
# shm_max_size_bytes: Optional - Maximum shared memory size in bytes.
#                     Default: 20MB (20 * 1024 * 1024 bytes)
#                     This prevents programs from exhausting shared memory

print("SandboxExecutor Parameters:")
print("- evaluate_worker: REQUIRED - The work object to execute")
print("- recur_kill_eval_proc: Optional - Recursively kill child processes")
print("- debug_mode: Optional - Enable debug logging (default: False)")
print("- join_timeout_seconds: Optional - Wait timeout in seconds (default: 10)")
print("- shm_max_size_bytes: Optional - Max shared memory (default: 20MB)")

## 3. Execution Methods

The main execution method `secure_execute` provides a safe way to run arbitrary code with resource controls.

In [ ]:
# secure_execute: Executes a method in an isolated process

# Parameters:
# worker_execute_method_name: Name of the method to invoke on the worker
# method_args: Positional arguments to pass to the method
# method_kwargs: Keyword arguments to pass to the method
# timeout_seconds: Maximum execution time before termination
#                  IMPORTANT: Set this appropriately for your use case
# redirect_to_devnull: If True, suppress all stdout/stderr output

# Returns: ExecutionResults (TypedDict)
# - result: The return value from the executed method
# - execution_time: Time taken to execute in seconds
# - error_msg: Error message if execution failed, None otherwise

print("secure_execute Return Values:")
print("- result: Any - The return value from the method")
print("- execution_time: float - Execution time in seconds")
print("- error_msg: str - Error message if failed, None otherwise")

## 4. Usage Examples

Let's explore various use cases from basic execution to complex integrations with the Evaluator.

In [ ]:
# Example 1: Create a Simple Worker and Execute in Sandbox

class SimpleWorker:
    """A simple computation worker demonstrating basic sandbox usage"""
    
    def add(self, a, b):
        """Returns the sum of two numbers"""
        return a + b
    
    def compute(self, n):
        """Computes the sum 1+2+...+n using Gauss's formula"""
        # Using formula: n*(n+1)/2 for efficiency
        return sum(range(1, n+1))

# Create worker and sandbox
worker = SimpleWorker()
sandbox = SandboxExecutor(
    evaluate_worker=worker,
    join_timeout_seconds=5,  # Wait up to 5 seconds for completion
    debug_mode=False
)

# Execute the 'add' method with positional arguments
result = sandbox.secure_execute(
    "add",
    method_args=[3, 5],
    timeout_seconds=2
)

print(f"Result: {result['result']}")
print(f"Execution time: {result['execution_time']}")
print(f"Error: {result['error_msg']}")

In [ ]:
# Example 2: Executing Methods with Keyword Arguments

# You can pass arguments using either positional (method_args)
# or keyword (method_kwargs) style

result = sandbox.secure_execute(
    "compute",
    method_kwargs={"n": 100},
    timeout_seconds=2
)

print(f"1+2+...+100 = {result['result']}")
print(f"Expected: 5050")

In [ ]:
# Example 3: Handling Timeout Scenarios

# Timeouts are critical for algorithm discovery - generated code may
# contain infinite loops or take too long to execute

class SlowWorker:
    """Simulates a long-running operation"""
    
    def slow_task(self, seconds):
        """Simulates a task that takes a fixed amount of time"""
        import time
        time.sleep(seconds)
        return "Done"

slow_worker = SlowWorker()
sandbox = SandboxExecutor(
    evaluate_worker=slow_worker,
    join_timeout_seconds=2  # Sandbox timeout: 2 seconds
)

# Execute a task that takes 3 seconds (will timeout)
# The sandbox will forcibly terminate the process after timeout_seconds
result = sandbox.secure_execute(
    "slow_task",
    method_args=[3],  # Task wants to sleep for 3 seconds
    timeout_seconds=2  # But we only allow 2 seconds max
)

print(f"Result: {result['result']}")
print(f"Error: {result['error_msg']}")
print(f"Execution time: {result['execution_time']}")

In [ ]:
# Example 4: Using Output Redirection

# Output redirection is useful when you want to:
# - Suppress verbose debugging output from executed code
# - Prevent information leakage through stdout/stderr
# - Reduce log noise in production environments

class OutputWorker:
    """A worker that produces output"""
    
    def print_hello(self):
        print("Hello from sandbox!")
        return "Done"

worker = OutputWorker()
sandbox = SandboxExecutor(
    evaluate_worker=worker,
    join_timeout_seconds=5
)

# Without redirection - output is visible
result = sandbox.secure_execute(
    "print_hello",
    redirect_to_devnull=False
)
print(f"Result: {result['result']}")

# With redirection - output is suppressed
result = sandbox.secure_execute(
    "print_hello",
    redirect_to_devnull=True
)
print(f"Result (output hidden): {result['result']}")

## 5. Security Best Practices

When using `SandboxExecutor` in production or algorithm discovery scenarios, follow these security guidelines to ensure safe execution of untrusted code.

In [ ]:
# Security Best Practices for SandboxExecutor

print("=== SECURITY BEST PRACTICES ===\n")

print("1. ALWAYS SET APPROPRIATE TIMEOUTS")
print("   - Never set timeout_seconds=0 or very high values")
print("   - For algorithm discovery: 2-10 seconds is usually sufficient")
print("   - For heavy computation: consider 30-60 seconds max\n")

print("2. USE PROCESS ISOLATION FOR UNTRUSTED CODE")
print("   - Always run code from LLM-generated candidates in sandbox")
print("   - Never disable process isolation for production use\n")

print("3. LIMIT SHARED MEMORY FOR RESOURCE CONSTRAINTS")
print("   - Default 20MB is suitable for most cases")
print("   - Increase only if your algorithms require more\n")

print("4. SET SANDBOX JOIN TIMEOUT APPROPRIATELY")
print("   - join_timeout_seconds should be > timeout_seconds")
print("   - This allows graceful cleanup after timeout triggers\n")

print("5. USE RECURSIVE KILL FOR RELIABLE CLEANUP")
print("   - Set recur_kill_eval_proc=True for untrusted code")
print("   - This ensures all child processes are terminated\n")

print("6. REDIRECT OUTPUT IN PRODUCTION")
print("   - Set redirect_to_devnull=True to prevent output-based")
print("     side-channel attacks or information leakage\n")

# Example: Creating a Secure Sandbox Configuration

def create_secure_sandbox(worker, max_timeout=10):
    """
    Creates a sandbox with security-hardened settings.
    
    Args:
        worker: The worker object to execute
        max_timeout: Maximum execution time in seconds (default: 10)
    
    Returns:
        Configured SandboxExecutor instance
    """
    return SandboxExecutor(
        evaluate_worker=worker,
        recur_kill_eval_proc=True,       # Ensure complete cleanup
        debug_mode=False,                # Disable debug in production
        join_timeout_seconds=max_timeout + 5,  # Allow cleanup time
        shm_max_size_bytes=20 * 1024 * 1024  # 20MB limit
    )

## 6. Integration with Evaluator

The `SandboxExecutor` integrates seamlessly with the `Evaluator` class to safely evaluate LLM-generated algorithm candidates. This is the primary use case in algorithm discovery workflows.

In [ ]:
# Example 5: Realistic Evaluator Integration

# This demonstrates how SandboxExecutor is used in production
# algorithm discovery workflows to evaluate generated candidates

class AlgorithmEvaluator:
    """
    Evaluates algorithm implementations against test cases.
    
    This is a realistic example showing how the sandbox is used
    in algorithm discovery to safely execute and evaluate code
    generated by LLMs.
    """
    
    def __init__(self, test_cases):
        """
        Initialize with test cases.
        
        Args:
            test_cases: List of tuples (input, expected_output)
        """
        self.test_cases = test_cases
    
    def evaluate(self, program_str):
        """
        Evaluates a program string against all test cases.
        
        Args:
            program_str: The algorithm implementation as a string
        
        Returns:
            Dictionary with score, execution_time, and error_msg
        """
        try:
            # In a real implementation, you would:
            # 1. Parse the program_str to extract the algorithm
            # 2. Execute it against each test case
            # 3. Calculate a score based on correctness and efficiency
            
            # Simulating evaluation for demonstration
            score = 0.95
            return {
                "score": score,
                "execution_time": 0.02,
                "error_msg": None
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)
            }

# Create evaluator and sandbox for algorithm discovery
test_cases = [
    ([1, 2, 3], [1, 2, 3]),           # Identity operation
    ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5])  # Sorting test
]
evaluator = AlgorithmEvaluator(test_cases)

# Create sandbox with secure settings for production use
sandbox = SandboxExecutor(
    evaluate_worker=evaluator,
    recur_kill_eval_proc=True,    # Ensure proper cleanup
    join_timeout_seconds=10,
    debug_mode=False
)

# Example LLM-generated algorithm candidate
candidate_program = """
def sort_array(arr):
    # Bubble sort implementation
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
"""

# Execute evaluation in sandbox - safe execution of untrusted code
result = sandbox.secure_execute(
    "evaluate",
    method_args=[candidate_program],
    timeout_seconds=5
)

print(f"Evaluation Result:")
print(f"  Score: {result['result']['score']}")
print(f"  Execution Time: {result['result']['execution_time']}s")
print(f"  Error: {result['result']['error_msg']}")

In [ ]:
# Example 6: Handling Malicious Code

# The sandbox protects against various malicious behaviors

class MaliciousWorker:
    """Simulates potentially malicious code for testing"""
    
    def attempt_infinite_loop(self):
        """Simulates an infinite loop attack"""
        while True:
            pass
    
    def attempt_memory_hog(self):
        """Simulates a memory exhaustion attack"""
        data = []
        while True:
            data.append([0] * 1000000)
    
    def attempt_subprocess_spawn(self):
        """Simulates spawning unauthorized child processes"""
        import subprocess
        subprocess.Popen(['sleep', '100'])
        return "Spawned process"

malicious_worker = MaliciousWorker()
sandbox = SandboxExecutor(
    evaluate_worker=malicious_worker,
    recur_kill_eval_proc=True,  # Recursive kill for child processes
    join_timeout_seconds=2,
    debug_mode=False
)

# Test 1: Infinite loop protection
result = sandbox.secure_execute(
    "attempt_infinite_loop",
    timeout_seconds=2
)
print(f"Infinite Loop Test:")
print(f"  Result: {result['result']}")
print(f"  Error: {result['error_msg']}")
print(f"  Execution time: {result['execution_time']}s\n")

In [ ]:
# Test 2: Memory exhaustion protection
result = sandbox.secure_execute(
    "attempt_memory_hog",
    timeout_seconds=2
)
print(f"Memory Hog Test:")
print(f"  Result: {result['result']}")
print(f"  Error: {result['error_msg']}")
print(f"  Execution time: {result['execution_time']}s\n")

In [ ]:
# Test 3: Subprocess spawning (may still succeed but process is isolated)
result = sandbox.secure_execute(
    "attempt_subprocess_spawn",
    timeout_seconds=2
)
print(f"Subprocess Spawn Test:")
print(f"  Result: {result['result']}")
print(f"  Error: {result['error_msg']}")
print(f"  Execution time: {result['execution_time']}s")

## 7. Summary

This guide covered the essential aspects of the `SandboxExecutor` for safe code execution in algorithm discovery workflows.

In [ ]:
# Quick Reference Summary

print("=== SANDBOX EXECUTOR QUICK REFERENCE ===\n")

print("| Component/Parameter      | Description |")
print("|--------------------------|-------------|")
print("| SandboxExecutor          | Initialize sandbox with worker |")
print("| secure_execute()         | Execute method in isolated process |")
print("| timeout_seconds          | Max execution time before termination |")
print("| redirect_to_devnull       | Suppress stdout/stderr output |")
print("| recur_kill_eval_proc      | Recursively kill child processes |")
print("| shm_max_size_bytes        | Limit shared memory usage |")

print("\nReturn Value (ExecutionResults):")
print("  - result: Return value from executed method")
print("  - execution_time: Time taken in seconds")
print("  - error_msg: Error message or None if successful")

print("\nBest Practices:")
print("  1. Always set appropriate timeouts")
print("  2. Use recur_kill_eval_proc=True for untrusted code")
print("  3. Set join_timeout_seconds > timeout_seconds")
print("  4. Use redirect_to_devnull=True in production")
print("  5. Follow the create_secure_sandbox() pattern for production")